# Bolo.ai — Orpheus-3B LoRA Fine-Tuning (Phase 3)

Fine-tunes `unsloth/orpheus-3b-0.1-ft` with LoRA on `data/metadata.csv` (2,104 rows: 1,352 English + 652 Hindi from `SPRINGLab/IndicTTS-*`, plus 100 self-recorded Hinglish clips — see `data/README.md` and `zdnd/decision.md` for how this dataset was built and why the Hindi text is transliterated to Latin script rather than Devanagari).

**Important note on audio tokenization (read before running):** Unsloth's TTS fine-tuning support may or may not auto-convert audio into Orpheus's discrete SNAC-based audio tokens depending on the exact library version — this notebook does **not** assume or depend on that. Instead it builds the token sequences manually, using the *exact same* control-token IDs and SNAC layer-offset scheme already verified end-to-end against the real `orpheus-3b-0.1-ft` checkpoint in Phase 2 (`inference/tts_server.py`). That means training labels and inference decoding are guaranteed to use the same scheme — no risk of a mismatch between how this notebook encodes audio and how the agent later decodes it.

**Run cells top to bottom. Do the small-scale sanity check (Section 4) before the full dataset preprocessing (Section 5) — don't skip ahead.**

## 1. Setup

In [ ]:
!git clone https://github.com/kodebyshubh/Bolo.ai.git /content/Bolo.ai 2>/dev/null || echo "already cloned"
%cd /content/Bolo.ai
!git pull

In [ ]:
!pip install -q -U unsloth snac soundfile librosa datasets indic-transliteration

**Restart the Colab runtime now** (Runtime → Restart session) after this install, same as in Phase 2 — then continue from the cell below.

In [ ]:
%cd /content/Bolo.ai
import sys
sys.path.insert(0, "/content/Bolo.ai")
!nvidia-smi

## 2. Load base model + LoRA

Same `load_in_4bit=False` (16-bit LoRA) and model name as the Phase 2 baseline, so the fine-tuned model stays directly comparable to it. LoRA config here is a standard, moderate setting for a ~3B Llama-architecture model — `r=16` is a common default that balances adapter capacity against the small dataset size (3 hours total, mostly a LoRA touch-up rather than heavy retraining).

In [ ]:
import torch
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/orpheus-3b-0.1-ft"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # let Unsloth pick -- falls back to float16 on T4 (confirmed in Phase 2)
    load_in_4bit=False,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
model.print_trainable_parameters()

## 3. Control tokens and SNAC audio encoding

These constants are imported directly from `inference/tts_server.py`, not re-typed here — the whole point is that training and inference must agree on the exact same token scheme. `audio_to_tokens()` below is the mathematical inverse of that file's `tokens_to_audio()`: it runs the SNAC encoder (audio → 3 hierarchical code layers) and re-interleaves them into Orpheus's 7-codes-per-frame sequence, exactly undoing the unpacking `tokens_to_audio()` does.

In [ ]:
from inference.tts_server import (
    START_OF_SPEECH, END_OF_SPEECH, START_OF_HUMAN, END_OF_HUMAN,
    START_OF_AI, AUDIO_TOKEN_OFFSET, TEXT_EOT, SAMPLE_RATE, tokens_to_audio,
)

print("START_OF_HUMAN", START_OF_HUMAN)
print("END_OF_HUMAN", END_OF_HUMAN)
print("START_OF_AI", START_OF_AI)
print("START_OF_SPEECH", START_OF_SPEECH)
print("END_OF_SPEECH", END_OF_SPEECH)
print("TEXT_EOT", TEXT_EOT)
print("AUDIO_TOKEN_OFFSET", AUDIO_TOKEN_OFFSET)

In [ ]:
from snac import SNAC
import soundfile as sf
import librosa

snac_model = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").eval()
if torch.cuda.is_available():
    snac_model = snac_model.to("cuda")


def audio_to_tokens(snac_model, audio_path):
    """Inverse of tts_server.tokens_to_audio(): encode a WAV file into the
    interleaved 7-codes-per-frame Orpheus audio-token sequence (as raw
    SNAC-offset integers, not yet shifted by AUDIO_TOKEN_OFFSET)."""
    array, sr = sf.read(audio_path, dtype="float32")
    if sr != SAMPLE_RATE:
        array = librosa.resample(array, orig_sr=sr, target_sr=SAMPLE_RATE)
    device = next(snac_model.parameters()).device
    waveform = torch.tensor(array, dtype=torch.float32).reshape(1, 1, -1).to(device)

    with torch.no_grad():
        codes = snac_model.encode(waveform)
    layer1 = codes[0].squeeze(0).squeeze(0).tolist()
    layer2 = codes[1].squeeze(0).squeeze(0).tolist()
    layer3 = codes[2].squeeze(0).squeeze(0).tolist()

    n_frames = len(layer1)
    tokens = []
    for i in range(n_frames):
        frame = [
            layer1[i],
            layer2[2 * i] + 4096 * 1,
            layer3[4 * i] + 4096 * 2,
            layer3[4 * i + 1] + 4096 * 3,
            layer2[2 * i + 1] + 4096 * 4,
            layer3[4 * i + 2] + 4096 * 5,
            layer3[4 * i + 3] + 4096 * 6,
        ]
        tokens.extend(AUDIO_TOKEN_OFFSET + v for v in frame)
    return tokens

## 4. Sanity check: encode → decode round trip (do this before anything else)

This is the single most important verification in this notebook. It does **not** touch the language model at all — it just checks that `audio_to_tokens()` (encode) and `tokens_to_audio()` (decode, already proven correct in Phase 2) are true inverses of each other. Take one real training clip, encode it, immediately decode it back, and listen. If it sounds like the original recording, the token scheme this notebook builds training labels with is correct. If it sounds wrong (garbled, wrong pitch, static), **stop here and debug this before running any training** — training on a broken encoding would silently waste the whole GPU budget.

In [ ]:
import csv
import os
from IPython.display import Audio, display

with open("data/metadata.csv", encoding="utf-8") as f:
    metadata_rows = list(csv.DictReader(f))
print(f"{len(metadata_rows)} rows in metadata.csv")

sample_row = metadata_rows[0]
sample_path = os.path.join("data/audio", sample_row["filename"])
print("testing round trip on:", sample_path, "->", repr(sample_row["text"]))

raw_tokens = audio_to_tokens(snac_model, sample_path)
print("encoded to", len(raw_tokens), "raw tokens")

# tokens_to_audio expects tokens already shifted into the model's vocab range
# and filters anything below AUDIO_TOKEN_OFFSET, so raw_tokens (already
# offset by audio_to_tokens) pass straight through.
reconstructed = tokens_to_audio(snac_model, raw_tokens)
sf.write("/content/roundtrip_test.wav", reconstructed, SAMPLE_RATE)
print("reconstructed duration:", len(reconstructed) / SAMPLE_RATE, "seconds")

print("\noriginal:")
display(Audio(sample_path))
print("reconstructed (encode -> decode round trip):")
display(Audio("/content/roundtrip_test.wav"))

**Listen to both clips above. They should sound essentially identical** (SNAC is lossy, so minor quality loss is expected, but the words, voice, and timing should match). Only proceed past this point once that's confirmed.

## 5. Build the training dataset

Each example: `[START_OF_HUMAN] + text_tokens + [TEXT_EOT, END_OF_HUMAN, START_OF_AI, START_OF_SPEECH] + audio_tokens + [END_OF_SPEECH]`. Loss is masked (`-100`) over the prompt portion (everything up through `START_OF_SPEECH`) so the model is only supervised on generating the audio tokens and the final `END_OF_SPEECH` -- exactly the portion `inference/tts_server.py` generates at inference time.

This preprocessing step runs SNAC encoding on all ~2,100 clips, which takes a while. It saves the tokenized dataset to disk (`training/tokenized_dataset/`) so a Colab disconnect doesn't lose this work -- re-running this cell loads the cached version instead of redoing it.

In [ ]:
from datasets import Dataset, load_from_disk

CACHE_DIR = "training/tokenized_dataset"


def build_example(row):
    text = row["text"]
    audio_path = os.path.join("data/audio", row["filename"])
    text_ids = tokenizer(text, return_tensors="pt").input_ids[0].tolist()
    audio_tokens = audio_to_tokens(snac_model, audio_path)

    prompt = [START_OF_HUMAN] + text_ids + [TEXT_EOT, END_OF_HUMAN, START_OF_AI, START_OF_SPEECH]
    completion = audio_tokens + [END_OF_SPEECH]
    input_ids = prompt + completion
    labels = [-100] * len(prompt) + completion
    return {"input_ids": input_ids, "labels": labels, "length": len(input_ids)}


if os.path.exists(CACHE_DIR):
    print(f"loading cached tokenized dataset from {CACHE_DIR}")
    tokenized_dataset = load_from_disk(CACHE_DIR)
else:
    examples = []
    skipped = 0
    for i, row in enumerate(metadata_rows):
        try:
            ex = build_example(row)
        except Exception as e:
            print(f"skipping row {i} ({row['filename']}): {e}")
            skipped += 1
            continue
        if ex["length"] > MAX_SEQ_LENGTH:
            skipped += 1
            continue
        examples.append(ex)
        if (i + 1) % 200 == 0:
            print(f"processed {i + 1}/{len(metadata_rows)}")

    print(f"built {len(examples)} examples, skipped {skipped} (errors or over {MAX_SEQ_LENGTH} tokens)")
    tokenized_dataset = Dataset.from_list(examples)
    tokenized_dataset.save_to_disk(CACHE_DIR)
    print(f"cached to {CACHE_DIR}")

print(tokenized_dataset)

## 6. Training arguments

`per_device_train_batch_size=1` with `gradient_accumulation_steps=4` (effective batch size 4) -- variable-length audio-token sequences make batching past size 1 awkward without a real padding collator, and this is a small LoRA touch-up, not large-scale training, so the extra complexity isn't worth it. `save_steps=100` guards against a Colab session drop mid-run (PRD risk: "Colab session timeout during training").

In [ ]:
from transformers import TrainingArguments, Trainer


def collate_fn(batch):
    assert len(batch) == 1, "this collator assumes per_device_train_batch_size=1"
    item = batch[0]
    input_ids = torch.tensor(item["input_ids"], dtype=torch.long).unsqueeze(0)
    labels = torch.tensor(item["labels"], dtype=torch.long).unsqueeze(0)
    attention_mask = torch.ones_like(input_ids)
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}


training_args = TrainingArguments(
    output_dir="training/checkpoints",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    optim="adamw_8bit",
    logging_steps=10,
    save_steps=100,
    save_total_limit=3,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=collate_fn,
)

## 7. Train

In [ ]:
trainer_stats = trainer.train()
print(trainer_stats)

## 8. Save the LoRA adapter

In [ ]:
FINAL_ADAPTER_DIR = "training/checkpoints/final_adapter"
model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
print(f"saved LoRA adapter to {FINAL_ADAPTER_DIR}")

## 9. Quick post-training sanity check

Same single-sentence check used throughout Phase 2, now against the fine-tuned model, before moving on to the full Phase 4 base-vs-finetuned evaluation.

In [ ]:
FastLanguageModel.for_inference(model)

from inference.tts_server import build_prompt

text = "Good afternoon, I wanted to check on my recent purchase."
input_ids = build_prompt(tokenizer, text).to(model.device)
with torch.no_grad():
    output_ids = model.generate(
        input_ids,
        max_new_tokens=1200,
        do_sample=False,
        repetition_penalty=1.3,
        eos_token_id=END_OF_SPEECH,
    )
generated = output_ids[0, input_ids.shape[1]:].tolist()
if END_OF_SPEECH in generated:
    generated = generated[: generated.index(END_OF_SPEECH)]
audio = tokens_to_audio(snac_model, generated)
sf.write("/content/finetuned_sanity_check.wav", audio, SAMPLE_RATE)
print("duration:", len(audio) / SAMPLE_RATE, "seconds")
display(Audio("/content/finetuned_sanity_check.wav"))